In [ ]:
import sys
sys.path.append('../')
from source import Process
import pandas as pd
import re
import json
import numpy as np

Use cd-hit multiple times decreasing the cluster similarity cutoff to check redundancy

In [ ]:
NcrdClusters = Process.ClusterCounter("../data/filtered/ncrd95.tagged.fasta")
ResfinderClusters = Process.ClusterCounter("../data/filtered/resfinder.tagged.fasta")
CardClusters = Process.ClusterCounter("../data/filtered/card.tagged.fasta")
MegaresClusters = Process.ClusterCounter("../data/filtered/megares.tagged.fasta")
HmdClusters = Process.ClusterCounter("../data/filtered/hmd.tagged.fasta")
NdaroClusters = Process.ClusterCounter("../data/filtered/ndaro.tagged.fasta")

In [ ]:
ClusterLimits = pd.DataFrame({
    "Database": ["NCRD", "Resfinder", "CARD", "MEGARES", "HMD", "NDARO"],
    "100%": [NcrdClusters[0][0], ResfinderClusters[0][0], CardClusters[0][0], MegaresClusters[0][0], HmdClusters[0][0], NdaroClusters[0][0]],
    "95%":  [NcrdClusters[0][1], ResfinderClusters[0][1], CardClusters[0][1], MegaresClusters[0][1], HmdClusters[0][1], NdaroClusters[0][1]],
    "90%":  [NcrdClusters[0][2], ResfinderClusters[0][2], CardClusters[0][2], MegaresClusters[0][2], HmdClusters[0][2], NdaroClusters[0][2]],
    "85%":  [NcrdClusters[0][3], ResfinderClusters[0][3], CardClusters[0][3], MegaresClusters[0][3], HmdClusters[0][3], NdaroClusters[0][3]],
    "80%":  [NcrdClusters[0][4], ResfinderClusters[0][4], CardClusters[0][4], MegaresClusters[0][4], HmdClusters[0][4], NdaroClusters[0][4]],  
    "75%":  [NcrdClusters[0][5], ResfinderClusters[0][5], CardClusters[0][5], MegaresClusters[0][5], HmdClusters[0][5], NdaroClusters[0][5]],
    "70%":  [NcrdClusters[0][6], ResfinderClusters[0][6], CardClusters[0][6], MegaresClusters[0][6], HmdClusters[0][6], NdaroClusters[0][6]],
})
ClusterLimits = ClusterLimits.set_index("Database").T
ClusterLimits.columns.name = None
ClusterLimits.to_csv("../data/processed/cdhitclusters.defaultsettings.csv", index=True, sep = "\t")

Check the shared proteins with identity above 95% among databases.

In [ ]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)

PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]
PairWiseAlignment = PairWiseAlignment.loc[PairWiseAlignment.pident >= 95]
PairWiseAlignment.replace({"RESFINDER":"ResFinder", "MEGARES":"MegaRes","HMDARG":"HMDARG-DB"}, inplace = True)
BlastPairWiseAlignmentPivoted = PairWiseAlignment.pivot_table(index="qseqtag", columns="sseqtag", values="pident", aggfunc="count").fillna(0)
BlastPairWiseAlignmentPivoted.index.name = None
BlastPairWiseAlignmentPivoted.columns.name = None
BlastPairWiseAlignmentPivoted.to_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov80.maxseq1.csv", index=True, sep = "\t")


In [ ]:
BlastPairWiseAlignmentPivoted

Standardize the resistance class of each protein

In [ ]:
CardIndex = pd.read_csv(
    "../data/raw/card.index.tsv",
    sep = "\t"
)
CardDict = CardIndex.drop_duplicates(subset="ARO Accession", keep='first').set_index("ARO Accession").to_dict(orient="index")
## Duplicates
CardIndex.value_counts("ARO Accession")[:5]

In [ ]:
CardIndex.loc[CardIndex["ARO Accession"] == "ARO:3003900"]

In [ ]:
NDAROIndex = pd.read_csv(
    "../data/raw/ndaro.index.tsv", 
    sep="\t")
NDAROIndex.fillna("Not Found", inplace=True)
NdaroDict = NDAROIndex.dropna(subset=["RefSeq protein"]).drop_duplicates(subset="RefSeq protein", keep='first').set_index("RefSeq protein").to_dict(orient="index")
NDAROIndex.value_counts("RefSeq protein")[:5]

In [ ]:
NDAROIndex.loc[NDAROIndex["RefSeq protein"] == "WP_000090315.1"]

In [ ]:
sorted(NDAROIndex.Class.unique())

In [ ]:
ResfinderIndex = pd.read_csv(
    "../data/raw/resfinder.phenotypes.tsv", 
    sep="\t")
ResfinderDict = ResfinderIndex.drop_duplicates(subset="Gene_accession no.", keep='first').set_index("Gene_accession no.").to_dict(orient="index")
ResfinderIndex.value_counts("Gene_accession no.")[:5]

In [ ]:
SCHEMAS = {
    "CARD": {
        "AROSplitPoint": 3,
        "IndexInfo": CardDict
    },
    "NDARO": {
        "AccSplitPoint": 0,
        "IndexInfo": NdaroDict
    },
    "MEGARES": {
        "DrugClassSplitPoint": 3,
        "NameSplitPoint": -1
    },
    "HMD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 5
    },
    "NCRD": {
        "DrugClassSplitPoint": 4,
        "NameSplitPoint": 2
    },
    "RESFINDER": {
        "DrugClassSplitPoint": -1,
        "NameSplitPoint": 1,
        "IndexInfo": ResfinderDict
    }
}

In [232]:
MetaDict = Process.CreateMetadataFile(
    "../data/filtered/AllDatabases.tagged.fasta_ids",
    SCHEMAS
    )


In [233]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov80.maxseq1.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
PairWiseAlignment["qcov"] = np.round((PairWiseAlignment["qend"] - PairWiseAlignment["qstart"] + 1) / (PairWiseAlignment["qlen"]) * 100, 2)
PairWiseAlignment["scov"] = np.round((PairWiseAlignment["send"] - PairWiseAlignment["sstart"] + 1) / (PairWiseAlignment["slen"]) * 100, 2)
PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]

In [234]:
PairWiseDictSource = PairWiseAlignment[["qseqid","full_qseq"]].set_index("qseqid").to_dict()["full_qseq"]
PairWiseDictTarget = PairWiseAlignment[["sseqid","full_sseq"]].set_index("sseqid").to_dict()["full_sseq"]

In [235]:
def MergeSequences(base, override1, override2):
    for key, value in base.items():
        try:     
            base[key].update({"Sequence": override1[key]})
            try:
                base[key].update({"Sequence": override2[key]})
            except:
                pass
        except:
            pass
    return base

In [236]:
MergeSequences(MetaDict, PairWiseDictSource, PairWiseDictTarget)

{'CARD_0': {'Drug Class': 'cephalosporin',
  'Name': 'CblA-1',
  'Mechanism': 'antibiotic inactivation',
  'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'},
 'CARD_1': {'Drug Class': 'cephalosporin;penicillin beta-lactam',
  'Name': 'SHV-52',
  'Mechanism': 'antibiotic inactivation',
  'Sequence': 'MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMDLASGRTLTAWRADERFPMISTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLAIVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'},
 'CARD_2': {'Drug Class': 'diaminopyrimidine antibiotic',
  'Name': 'dfrF',
  'Mechanism': 'antibiotic target replacement',
  'Sequence': 'MIGLIVAR

In [237]:
def CreateGeneralClass(base):
    for key, value in base.items():
        DrugClass = base[key]["Drug Class"]
        DrugClass = DrugClass.replace("antibiotic", "").lower().strip()
        DrugClass = DrugClass.replace("betalactam","beta-lactam")
        DrugClass = DrugClass.replace("beta_lactam","beta-lactam")
        base[key].update({"Drug Class": DrugClass.rstrip('s')})
        try:
            if DrugClass.endswith("beta-lactam") and value["Mechanism"] != "antibiotic efflux":
                pass
            if value["Mechanism"].strip() == "antibiotic inactivation" or value["Mechanism"].strip() == "permeability to antibiotic" or value["Mechanism"] == "antibiotic target replacement":
                if "cephalosporin" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
                if "carbapenem" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
                if "penicillin" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
                if "beta_lactam" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
                if "betalactams" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
                if "penicillin" in DrugClass:
                    base[key].update({"Drug Class": "beta-lactam"})
        except:
            pass
        Macrolide = bool(re.search(rf'\b{re.escape("macrolide")}\b', DrugClass))
        Lincosamide = bool(re.search(rf'\b{re.escape("lincosamide")}\b', DrugClass))
        Streptogramin = bool(re.search(rf'\b{re.escape("streptogramin")}\b', DrugClass))
        if Macrolide and Lincosamide and Streptogramin:
            base[key].update({"Drug Class": "MLS"})


    return base

In [250]:
MetaDict["NCRD_3359"]

{'Drug Class': 'glycopeptide',
 'Name': 'PmrF',
 'Sequence': 'MFEIHPVKKVSVVIPVYNEQESLPELIRRTTTACESLGKEYEILLIDDGSSDNSAHMLVEASQAENSHIVSILLNRNYGQHSAIMAGFSHVTGDLIITLDADLQNPPEEIPRLVAKADEGYDVVGTVRQNRRDSWFRKTASKMINRLIQRTTGKAMGDYGCMLRAYRRHIVDAMLHCHERSTFIPILANIFARRAIEIPVHHSEREFGESKYSFMRLINLMYDLVTCLTTTPLRMLSLLGSIIAIGGFSIAVLLVILRLTFGPQWAAEGVFMLFAVLFTFIGAQFIGMGLLGEYIGRDLHRCPRPPPAIFVQQVIRPSSKENE'}

In [239]:
MetaDict["CARD_1990"]

{'Drug Class': 'fluoroquinolone antibiotic;macrolide antibiotic;penicillin beta-lactam',
 'Name': 'gadW',
 'Mechanism': 'antibiotic efflux',
 'Sequence': 'MAHVCSVILVRRSFDIHHEQQKISLHNESILLLDKNLADDFAFCSLDTRRLDIEELTVCHYLQNIRQLPRNLGLHSKDRLLINQSPPIQLVTAIFDSFNDPRVNSPILSKMLYLSCLSMFSHKKELIPLLFNSISTVSGKVERLISFDIAKRWYLRDIAERMYTSESLIKKKLQDENTCFSKILLASRMSMARRLLELRQIPLHTIAEKCGYSSTSYFINTFRQYYGVTPHQFSQHSPGTFS'}

In [240]:
MetaDict["CARD_686"]

{'Drug Class': 'cephalosporin;fluoroquinolone antibiotic;fusidane antibiotic;macrolide antibiotic',
 'Name': 'cmeB',
 'Mechanism': 'antibiotic efflux',
 'Sequence': 'MFSKFFIERPVFASVVAIIISLAGVIGLTNLPIEQYPSLTPPTVKVSATYTGADAQTIASTVASPIEDAINGADNMIYMDSTSSSSGTMSLTVYFDIGTDPDQATIDVNNRISAATAKMPDAVKKLGVTVRKTSSATLAAISMYSSDGSMSAVDVYNYIALNVLDELKRVPGVGDANAIGNRNYSLRIWLKPDLLNKFKITATDVISAVNDQNAQYATGKIGEEPVTQKSPYVYSITMQGRLQNPSEFENIILRTNDDGSFLRLKDIADVEIGSQQYSSQGRLNGNDAVPIIINLQSGANALHTAELVQAKMQELSKNFPKGLTYKIPYDTTKFVIESIKEVIKTFIEALILVIIVMYMFLKNFRATLIPMIAVPVSLLGTFAGLYVLGFSINLLTLFALILAIGIVVDDAIIVVENIDRILHENEQISVKDAAIQAMQEVSSPVISIVLVLCAVFIPVSFISGFVGEIQRQFALTLAISVTISGFVALTLTPSLCALFLRRNEGEPFKFVRKFNDFFDWSTSVFSAGVAYILKRTIRFVLIFCIMLGTIFYLNKAVPNSLVPEEDQGLMIGIINLPSASALHRTISEVDHISQEVLKTNGIKDAMAMIGFDLFTSSLKENAAAMFIGLQDWKDRNVSADKIAMELNKKFAFDRNASSIFIGLPPIPGLSITGGFEMYVQNKSGKSYDQIQKDVNKLVAAANQRKELSRVRTTLDTTFPQYKLIIDRDKLKHYNLNMQDVFNTMNATIGTYYVNDFSMLGKNFQVNIRAKGDFRNTQDALKNIFVRSNDGKMIPLDSFLTLQRSSGPDDVKRFNLFPAAQVQGQPAPGYTSGQA

In [241]:
MetaDict["CARD_942"]

{'Drug Class': 'cephalosporin;penicillin beta-lactam',
 'Name': 'SHV-1',
 'Mechanism': 'antibiotic inactivation',
 'Sequence': 'MRYIRLCIISLLATLPLAVHASPQPLEQIKLSESQLSGRVGMIEMDLASGRTLTAWRADERFPMMSTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLATVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'}

In [242]:
MetaDict = CreateGeneralClass(MetaDict)

In [243]:
MetaDict["CARD_1990"]

{'Drug Class': 'fluoroquinolone ;macrolide ;penicillin beta-lactam',
 'Name': 'gadW',
 'Mechanism': 'antibiotic efflux',
 'Sequence': 'MAHVCSVILVRRSFDIHHEQQKISLHNESILLLDKNLADDFAFCSLDTRRLDIEELTVCHYLQNIRQLPRNLGLHSKDRLLINQSPPIQLVTAIFDSFNDPRVNSPILSKMLYLSCLSMFSHKKELIPLLFNSISTVSGKVERLISFDIAKRWYLRDIAERMYTSESLIKKKLQDENTCFSKILLASRMSMARRLLELRQIPLHTIAEKCGYSSTSYFINTFRQYYGVTPHQFSQHSPGTFS'}

In [244]:
MetaDict["CARD_686"]

{'Drug Class': 'cephalosporin;fluoroquinolone ;fusidane ;macrolide',
 'Name': 'cmeB',
 'Mechanism': 'antibiotic efflux',
 'Sequence': 'MFSKFFIERPVFASVVAIIISLAGVIGLTNLPIEQYPSLTPPTVKVSATYTGADAQTIASTVASPIEDAINGADNMIYMDSTSSSSGTMSLTVYFDIGTDPDQATIDVNNRISAATAKMPDAVKKLGVTVRKTSSATLAAISMYSSDGSMSAVDVYNYIALNVLDELKRVPGVGDANAIGNRNYSLRIWLKPDLLNKFKITATDVISAVNDQNAQYATGKIGEEPVTQKSPYVYSITMQGRLQNPSEFENIILRTNDDGSFLRLKDIADVEIGSQQYSSQGRLNGNDAVPIIINLQSGANALHTAELVQAKMQELSKNFPKGLTYKIPYDTTKFVIESIKEVIKTFIEALILVIIVMYMFLKNFRATLIPMIAVPVSLLGTFAGLYVLGFSINLLTLFALILAIGIVVDDAIIVVENIDRILHENEQISVKDAAIQAMQEVSSPVISIVLVLCAVFIPVSFISGFVGEIQRQFALTLAISVTISGFVALTLTPSLCALFLRRNEGEPFKFVRKFNDFFDWSTSVFSAGVAYILKRTIRFVLIFCIMLGTIFYLNKAVPNSLVPEEDQGLMIGIINLPSASALHRTISEVDHISQEVLKTNGIKDAMAMIGFDLFTSSLKENAAAMFIGLQDWKDRNVSADKIAMELNKKFAFDRNASSIFIGLPPIPGLSITGGFEMYVQNKSGKSYDQIQKDVNKLVAAANQRKELSRVRTTLDTTFPQYKLIIDRDKLKHYNLNMQDVFNTMNATIGTYYVNDFSMLGKNFQVNIRAKGDFRNTQDALKNIFVRSNDGKMIPLDSFLTLQRSSGPDDVKRFNLFPAAQVQGQPAPGYTSGQAIEAIAQVAKETLGDDYSIAWSGSAYQEVSSK

In [245]:
MetaDict["CARD_0"]

{'Drug Class': 'beta-lactam',
 'Name': 'CblA-1',
 'Mechanism': 'antibiotic inactivation',
 'Sequence': 'MKAYFIAILTLFTCIATVVRAQQMSELENRIDSLLNGKKATVGIAVWTDKGDMLRYNDHVHFPLLSVFKFHVALAVLDKMDKQSISLDSIVSIKASQMPPNTYSPLRKKFPDQDFTITLRELMQYSISQSDNNACDILIEYAGGIKHINDYIHRLSIDSFNLSETEDGMHSSFEAVYRNWSTPSAMVRLLRTADEKELFSNKELKDFLWQTMIDTETGANKLKGMLPAKTVVGHKTGSSDRNADGMKTADNDAGLVILPDGRKYYIAAFVMDSYETDEDNANIIARISRMVYDAMR'}

In [246]:
MetaDict["CARD_942"]

{'Drug Class': 'beta-lactam',
 'Name': 'SHV-1',
 'Mechanism': 'antibiotic inactivation',
 'Sequence': 'MRYIRLCIISLLATLPLAVHASPQPLEQIKLSESQLSGRVGMIEMDLASGRTLTAWRADERFPMMSTFKVVLCGAVLARVDAGDEQLERKIHYRQQDLVDYSPVSEKHLADGMTVGELCAAAITMSDNSAANLLLATVGGPAGLTAFLRQIGDNVTRLDRWETELNEALPGDARDTTTPASMAATLRKLLTSQRLSARSQRQLLQWMVDDRVAGPLIRSVLPAGWFIADKTGAGERGARGIVALLGPNNKAERIVVIYLRDTPASMAERNQQIAGIGAALIEHWQR'}

In [247]:
Classes = pd.Series([MetaDict[key]["Drug Class"] for key, value in MetaDict.items()])

In [248]:
Classes.value_counts().head(10)

beta-lactam        26821
multidrug          13319
glycopeptide        8151
aminoglycoside      4943
bacitracin          4228
tetracycline        2932
MLS                 2655
na                  1290
chloramphenicol     1137
polymyxin           1114
Name: count, dtype: int64

In [249]:
with open("../data/processed/MetaDict.cov80.maxseq1.json", "w+") as json_file  :
    json.dump(MetaDict, json_file, indent=4)